# Model Building for Sales Forecasting
This project aims to build a robust sales forecasting system using historical sales data. The objective is to accurately predict future sales to support business decisions such as inventory planning, demand forecasting, and resource allocation. Various machine learning, statistical time series, and deep learning models are implemented and compared to identify the most effective approach.

The required libraries for data processing, model building, evaluation, and time series forecasting are imported below.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger('cmdstanpy').disabled = True
pd.options.display.float_format = '{:.2f}'.format

The feature-engineered dataset is loaded, which includes lag features, rolling statistics, and time-based variables created in the previous step.

In [2]:
df = pd.read_csv(r"C:\Users\Aniket\Downloads\Capstone Projects\Sales Forecasting & Demand Prediction System\data\df_feature_enigneered.csv")
df["Date"] = pd.to_datetime(df["Date"])
df.head()

,Date,Sales,year,month,dayofweek,lag_1,lag_3,lag_7,lag_14,lag_30,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_7,day,quarter,is_weekend,is_month_start,is_month_end
0,2015-02-14,576.73,2015,2,5,129.57,14.56,97.11,1097.25,16.45,418.55,404.96,593.87,722.89,14,1,1,0,0
1,2015-02-15,21.36,2015,2,6,576.73,2043.40,134.38,426.67,288.06,487.07,367.78,612.55,709.96,15,1,1,0,0
2,2015-02-16,9.04,2015,2,0,21.36,129.57,330.51,3.93,19.54,470.92,338.83,603.66,720.53,16,1,0,0,0
3,2015-02-17,54.21,2015,2,1,9.04,576.73,180.32,240.50,4407.10,425.00,339.19,603.31,740.92,17,1,0,0,0
4,2015-02-18,37.78,2015,2,2,54.21,21.36,14.56,290.67,87.16,406.98,325.89,458.21,749.35,18,1,0,0,0


The dataset is sorted chronologically by date to maintain the temporal sequence, which is essential for time series modeling.

In [3]:
df = df.sort_values("Date")
df.reset_index(drop=True,inplace=True)

The dataset is split into training and testing sets based on time. Data before 2018 is used for training, while data from 2018 onwards is used for testing to simulate real-world forecasting.

In [4]:
train = df[df["Date"] < "2018-01-01"]
test = df[df["Date"] >= "2018-01-01"]
print(train.shape,test.shape)

(877, 19) (322, 19)


The target variable `Sales` is separated from the feature set. The `Date` column is excluded as it is not directly used by machine learning models.

In [5]:
x_train = train.drop(columns=["Sales","Date"])
y_train = train["Sales"]
x_test = test.drop(columns=["Sales","Date"])
y_test = test["Sales"]

A baseline model is created using the previous day's sales (`lag_1`) as the prediction. This provides a reference point to evaluate whether advanced models improve performance.

## Baseline model :

In [6]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
y_pred_baseline = test["lag_1"]
mae_baseline = mean_absolute_error(y_test,y_pred_baseline)
mse_baseline = mean_squared_error(y_test,y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
print("Baseline MAE:",mae_baseline)
print("Baseline MSE:",mse_baseline)
print("Baseline RMSE:",rmse_baseline)

Baseline MAE: 2301.0476850931677
Baseline MSE: 10492617.83026591
Baseline RMSE: 3239.2310553997086


Multiple machine learning models are defined and trained to compare performance across different algorithms, including linear models, tree-based models, and ensemble methods.

## Machine learning Models :

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
 
models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "XGBoost": XGBRegressor(random_state=42,verbosity=0),
    "LightGBM": LGBMRegressor(random_state=42,verbose=-1),
    "CatBoost": CatBoostRegressor(random_state=42,verbose=0)}

Each model is trained on the training dataset and evaluated on the test dataset using MAE, MSE, and RMSE metrics to measure prediction accuracy.

In [8]:
results = []
for name, model in models.items():
    print("\n", name.upper())
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test,y_pred)
    rmse = np.sqrt(mse)
    results.append((name, mae, mse, rmse))
    print("MAE:",mae)
    print("MSE:",mse)
    print("RMSE:",rmse)


 LINEARREGRESSION
MAE: 1585.0125008847456
MSE: 5397733.599714448
RMSE: 2323.3023048485206

 DECISIONTREE
MAE: 1969.4093894409937
MSE: 8293227.846447639
RMSE: 2879.796493929326

 RANDOMFOREST
MAE: 1651.7042721335404
MSE: 5710447.536039195
RMSE: 2389.6542712365726

 GRADIENTBOOSTING
MAE: 1628.062665835994
MSE: 5952986.74742104
RMSE: 2439.874330251671

 KNN
MAE: 1772.1017849068323
MSE: 6424390.129409082
RMSE: 2534.638066748206

 SVR
MAE: 1657.5497044372742
MSE: 7291082.199244203
RMSE: 2700.200399830391

 XGBOOST
MAE: 1898.2286784965277
MSE: 6873145.017244499
RMSE: 2621.668365229382

 LIGHTGBM
MAE: 1732.8834607450121
MSE: 6062957.0429221215
RMSE: 2462.3072600555197

 CATBOOST
MAE: 1659.3518405967718
MSE: 5910099.865371034
RMSE: 2431.0696957041428


In [9]:
results_df = pd.DataFrame(results, columns=["Model", "MAE", "MSE", "RMSE"])
results_df = results_df.sort_values(by="RMSE")
results_df

,Model,MAE,MSE,RMSE
0,LinearRegression,1585.01,5397733.60,2323.30
2,RandomForest,1651.70,5710447.54,2389.65
8,CatBoost,1659.35,5910099.87,2431.07
3,GradientBoosting,1628.06,5952986.75,2439.87
7,LightGBM,1732.88,6062957.04,2462.31
4,KNN,1772.10,6424390.13,2534.64
6,XGBoost,1898.23,6873145.02,2621.67
5,SVR,1657.55,7291082.20,2700.20
1,DecisionTree,1969.41,8293227.85,2879.80


Based on the initial model comparison, Linear Regression and Random Forest emerged as the top-performing models with the lowest error metrics. Linear Regression performed well due to the presence of strong linear relationships created through lag and rolling features, while Random Forest captured non-linear patterns effectively. These two models are selected for further improvement through tuning techniques.

## Tunning Machine Learning Models : 

### Tunned Linear Regression / Ridge

To improve the performance of Linear Regression, Ridge Regression is applied as a regularization technique. Ridge helps reduce overfitting by penalizing large coefficients, making the model more stable and generalizable.

In [10]:
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)
y_pred_ridge = ridge.predict(x_test)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mse_ridge)
print("Ridge MAE:", mae_ridge)
print("Ridge MSE:", mse_ridge)
print("Ridge RMSE:", rmse_ridge)

Ridge MAE: 1585.0206696226765
Ridge MSE: 5397129.272829494
Ridge RMSE: 2323.172243470013


###  Tunned Random Forest

Hyperparameter tuning is performed using GridSearchCV with TimeSeriesSplit to optimize the Random Forest model while preserving temporal order.

In [11]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]}

In [12]:
rf = RandomForestRegressor(random_state=42)
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1)
grid.fit(x_train, y_train)
best_rf = grid.best_estimator_
print(grid.best_params_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}


In [13]:
y_pred_rf = best_rf.predict(x_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test,  y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
 

print("Tuned RF MAE:", mae_rf)
print("Tuned RF MSE:", mse_rf)
print("Tuned RF RMSE:", rmse_rf)

Tuned RF MAE: 1600.4357421310954
Tuned RF MSE: 5625337.497570196
Tuned RF RMSE: 2371.7793947941695


## Time Series Models : 

### Prophet

The Prophet model is used to capture trend and seasonality components in the time series data and generate future forecasts.

In [14]:
df_prophet = df[["Date", "Sales"]]
df_prophet.columns = ["ds", "y"]

In [15]:
from prophet import Prophet
model = Prophet()
model.fit(df_prophet)
future = model.make_future_dataframe(periods=322) 
forecast = model.predict(future)

In [16]:
y_true_ts = df_prophet["y"][-322:].values
y_pred_ts = forecast["yhat"][-322:].values
mae_ts = mean_absolute_error(y_true_ts, y_pred_ts)
mse_ts = mean_squared_error(y_true_ts, y_pred_ts)
rmse_ts = np.sqrt(mse_ts)
print("Prophet MAE:", mae_ts)
print("Prophet MSE:", mse_ts)
print("Prophet RMSE:", rmse_ts)

Prophet MAE: 1730.5807420640033
Prophet MSE: 5562767.628831083
Prophet RMSE: 2358.552019530433


### ARIMA

ARIMA (AutoRegressive Integrated Moving Average) is used as a traditional statistical method for time series forecasting, capturing linear dependencies in the data.

In [17]:
from statsmodels.tsa.arima.model import ARIMA
arima_model = ARIMA(df["Sales"], order=(5,1,0))
arima_model_fit = arima_model.fit()
arima_forecast = arima_model_fit.forecast(steps=len(test))
mae_arima = mean_absolute_error(y_test, arima_forecast)
mse_arima = mean_squared_error(y_test, arima_forecast)
rmse_arima = np.sqrt(mse_arima)
print("ARIMA MAE:", mae_arima)
print("ARIMA MSE:", mse_arima)
print("ARIMA RMSE:", rmse_arima)

ARIMA MAE: 1606.33686395343
ARIMA MSE: 6348246.891640102
ARIMA RMSE: 2519.572759743227


### SARIMA

SARIMA extends ARIMA by incorporating seasonality, allowing the model to capture repeating patterns over time.

In [18]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
sarima_model = SARIMAX(df["Sales"], order=(1,1,1), seasonal_order=(1,1,1,7))
sarima_fit = sarima_model.fit(disp=False)
sarima_forecast = sarima_fit.forecast(steps=len(test))
mae_sarima = mean_absolute_error(y_test, sarima_forecast)
mse_sarima = mean_squared_error(y_test, sarima_forecast)
rmse_sarima = np.sqrt(mse_sarima)
print("SARIMA MAE:", mae_sarima)
print("SARIMA MSE:", mse_sarima)
print("SARIMA RMSE:", rmse_sarima)

SARIMA MAE: 2146.634474790111
SARIMA MSE: 6593244.067238623
SARIMA RMSE: 2567.7313074460544


## Deep Learning Models

### LSTM

LSTM (Long Short-Term Memory) is implemented to capture sequential dependencies in the data using deep learning.

In [19]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[["Sales"]])

In [20]:
def create_sequences(data, seq_length=30):
    x, y = [], []
    for i in range(len(data) - seq_length):
        x.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(x), np.array(y)
x, y = create_sequences(scaled_data, 30)

In [21]:
split = int(0.8 * len(x))
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
model = Sequential()
model.add(LSTM(50, activation='tanh', input_shape=(30,1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.fit(x_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
30/30 [==============================] - 3s 14ms/step - loss: 0.0058
Epoch 2/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 3/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 4/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 5/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0053
Epoch 6/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 7/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 8/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 9/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 10/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0053


In [23]:
y_pred = model.predict(x_test)
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(y_pred)
mae_lstm = mean_absolute_error(y_test_inv, y_pred_inv)
mse_lstm = mean_squared_error(y_test_inv, y_pred_inv)
rmse_lstm = np.sqrt(mse_lstm)
print("LSTM MAE:", mae_lstm)
print("LSTM MSE:", mse_lstm)
print("LSTM RMSE:", rmse_lstm)

8/8 [==============================] - 1s 5ms/step
LSTM MAE: 1732.7648508179755
LSTM MSE: 6224672.2708100695
LSTM RMSE: 2494.9293117862217


### GRU

GRU (Gated Recurrent Unit) is a simplified version of LSTM that captures temporal patterns with fewer parameters and faster training

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
model = Sequential()
model.add(GRU(50, activation='relu', input_shape=(30,1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.fit(x_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
30/30 [==============================] - 3s 14ms/step - loss: 0.0062
Epoch 2/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 3/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 4/10
30/30 [==============================] - 0s 13ms/step - loss: 0.0054
Epoch 5/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 6/10
30/30 [==============================] - 0s 13ms/step - loss: 0.0054
Epoch 7/10
30/30 [==============================] - 0s 13ms/step - loss: 0.0055
Epoch 8/10
30/30 [==============================] - 0s 14ms/step - loss: 0.0054
Epoch 9/10
30/30 [==============================] - 0s 13ms/step - loss: 0.0054
Epoch 10/10
30/30 [==============================] - 0s 13ms/step - loss: 0.0054


In [25]:
y_pred = model.predict(x_test)
y_test_gru = scaler.inverse_transform(y_test)
y_pred_gru = scaler.inverse_transform(y_pred)
mae_gru = mean_absolute_error(y_test_gru, y_pred_gru)
mse_gru = mean_squared_error(y_test_gru, y_pred_gru)
rmse_gru = np.sqrt(mse_gru)
print("GRU MAE:", mae_gru)
print("GRU MSE:", mse_gru)
print("GRU RMSE:", rmse_gru)

8/8 [==============================] - 0s 4ms/step
GRU MAE: 1756.5799711354834
GRU MSE: 6272813.812235262
GRU RMSE: 2504.5586062688294


## Final Model Comparsion

The performance of all models is compared using evaluation metrics to identify the most effective approach for sales forecasting.

In [26]:
final_results = pd.DataFrame({"Model": ["Tuned RandomForest", "Ridge", "Prophet", "ARIMA", "SARIMA", "LSTM", "GRU"],
                              "MAE": [mae_rf, mae_ridge, mae_ts, mae_arima, mae_sarima, mae_lstm, mae_gru],
                              "MSE": [mse_rf, mse_ridge, mse_ts, mse_arima, mse_sarima, mse_lstm, mse_gru],
                              "RMSE": [rmse_rf, rmse_ridge, rmse_ts, rmse_arima, rmse_sarima, rmse_lstm, rmse_gru]})
final_results = final_results.sort_values(by="RMSE")
final_results

,Model,MAE,MSE,RMSE
1,Ridge,1585.02,5397129.27,2323.17
2,Prophet,1730.58,5562767.63,2358.55
0,Tuned RandomForest,1600.44,5625337.50,2371.78
5,LSTM,1732.76,6224672.27,2494.93
6,GRU,1756.58,6272813.81,2504.56
3,ARIMA,1606.34,6348246.89,2519.57
4,SARIMA,2146.63,6593244.07,2567.73


From the final comparison, Ridge Regression achieved the lowest error metrics among all models, making it the best-performing model for this dataset. This indicates that the relationships between features and target variable are largely linear, and regularization helped in improving model stability and reducing overfitting.While advanced models such as Random Forest, ARIMA, SARIMA, and deep learning models like LSTM and GRU were able to capture complex patterns, they did not outperform the regularized linear model. This suggests that the engineered features such as lag variables and rolling statistics were highly effective in transforming the problem into a form suitable for linear modeling.Therefore,Ridge Regression is selected as the final model for deployment.

In [27]:
import joblib
joblib.dump(ridge,"final_model.pkl")

['final_model.pkl']

# Conclusion:
In this project, multiple approaches including machine learning models, time series techniques, and deep learning models were explored for sales forecasting. Feature engineering played a crucial role in improving model performance by capturing temporal patterns through lag features and rolling statistics.
Among all models, Ridge Regression achieved the best performance, indicating that the problem could be effectively modeled using a linear approach with regularization. While time series and deep learning models captured complex patterns, they did not outperform the simpler and more efficient regularized model.
The final selected model can be used for real-world forecasting and can be further enhanced by incorporating external factors such as promotions, holidays, and economic indicators.
# Business Recommendations :
Based on the forecasting results, the business can leverage the model to improve inventory planning by maintaining optimal stock levels and reducing both overstocking and stockouts. Accurate sales predictions enable better demand planning, allowing the company to align procurement and supply chain operations more efficiently.
The model can also support promotional strategies by identifying periods of high and low demand. During low-demand periods, targeted marketing campaigns and discounts can be introduced to boost sales, while during peak periods, pricing strategies can be optimized to maximize revenue.
Additionally, the forecasts can assist in workforce and resource planning by anticipating demand fluctuations. This helps in allocating staff and logistics resources more effectively, reducing operational inefficiencies.
Finally, the model can be integrated into a real-time dashboard to continuously monitor sales trends and update predictions, enabling data-driven decision-making at both strategic and operational levels.